# 🌾 Crop Recommendation and Yield Prediction Using Machine Learning

This notebook provides an end-to-end exploratory data analysis (EDA), model training, benchmarking, user-defined inputs, and inference pipeline for:
1. **Stage 1 (Classification)**: Multi-class crop variety recommendation based on soil macronutrients (N, P, K, pH) and meteorological parameters (Temperature, Humidity, Rainfall).
2. **Stage 2 (Regression)**: Supervised continuous yield estimation (Tonnes/Ha) and total farm harvest calculation.
3. **User-Defined & Batch Tools**: Interactive and batch DataFrame prediction pipelines for custom farm measurements.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
    r2_score, mean_squared_error, mean_absolute_error
)

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import Ridge, LinearRegression

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print("Libraries loaded successfully!")

## 1. Exploratory Data Analysis: Crop Recommendation Dataset

In [ ]:
rec_df = pd.read_csv('../data/Crop_recommendation.csv')
print("Shape:", rec_df.shape)
print("Unique crops:", rec_df['label'].nunique())
rec_df.head()

In [ ]:
rec_df.describe()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(rec_df.drop('label', axis=1).corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Correlation Matrix of Soil and Climatic Features", fontsize=13, fontweight='bold')
plt.show()

## 2. Stage 1: Crop Classification Benchmarking

In [ ]:
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
X = rec_df[features]
le = LabelEncoder()
y = le.fit_transform(rec_df['label'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

classifiers = {
    'Random Forest': (RandomForestClassifier(n_estimators=120, random_state=42), False),
    'Decision Tree': (DecisionTreeClassifier(max_depth=12, random_state=42), False),
    'Gaussian Naive Bayes': (GaussianNB(), True),
    'SVM (RBF)': (SVC(kernel='rbf', C=10.0, probability=True, random_state=42), True),
    'KNN': (KNeighborsClassifier(n_neighbors=5), True),
    'Gradient Boosting': (GradientBoostingClassifier(n_estimators=100, random_state=42), False)
}

clf_results = []
for name, (model, scale_req) in classifiers.items():
    xtr = X_train_scaled if scale_req else X_train
    xte = X_test_scaled if scale_req else X_test
    model.fit(xtr, y_train)
    preds = model.predict(xte)
    clf_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds, average='weighted'),
        'Recall': recall_score(y_test, preds, average='weighted'),
        'F1_Score': f1_score(y_test, preds, average='weighted')
    })

clf_df = pd.DataFrame(clf_results).sort_values(by='F1_Score', ascending=False)
clf_df

## 3. Stage 2: Crop Yield Regression Benchmarking

In [ ]:
yield_df = pd.read_csv('../data/crop_yield.csv')
print("Yield Data Shape:", yield_df.shape)
yield_df.head()

In [ ]:
cat_cols = ['Crop']
num_cols = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'Area_ha']

X_y = yield_df[cat_cols + num_cols]
y_y = yield_df['Yield_tonnes_per_ha']

X_y_train, X_y_test, y_y_train, y_y_test = train_test_split(X_y, y_y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', StandardScaler(), num_cols)
])

regressors = {
    'Random Forest Regressor': RandomForestRegressor(n_estimators=120, random_state=42),
    'Gradient Boosting Regressor': GradientBoostingRegressor(n_estimators=120, random_state=42),
    'Extra Trees Regressor': ExtraTreesRegressor(n_estimators=120, random_state=42),
    'Ridge': Ridge(alpha=1.0),
    'Linear Regression': LinearRegression()
}

reg_results = []
for name, reg in regressors.items():
    pipe = Pipeline([('prep', preprocessor), ('reg', reg)])
    pipe.fit(X_y_train, y_y_train)
    preds = pipe.predict(X_y_test)
    reg_results.append({
        'Model': name,
        'R2_Score': r2_score(y_y_test, preds),
        'RMSE': np.sqrt(mean_squared_error(y_y_test, preds)),
        'MAE': mean_absolute_error(y_y_test, preds)
    })

reg_df = pd.DataFrame(reg_results).sort_values(by='R2_Score', ascending=False)
reg_df

## 4. End-to-End Prediction Verification

In [ ]:
import sys
sys.path.append('..')
from predict import CropAndYieldPredictor

engine = CropAndYieldPredictor(models_dir='../models')
# Test Rice environment
output = engine.predict_all(n=85, p=48, k=40, temp=24.0, hum=84.0, ph=6.4, rain=240.0, area_ha=3.0)
print("Stage 1 Recommendation:", output['stage1_recommendation'])
print("Stage 2 Yield Forecast:", output['stage2_yield_forecast'])


## 5. User-Defined Custom Prediction & Batch Farm Processing

In [ ]:
# User-defined custom inputs simulation
user_farm = {
    'n': 92.0, 'p': 46.0, 'k': 42.0,
    'temp': 25.0, 'hum': 82.0, 'ph': 6.6, 'rain': 230.0,
    'area_ha': 4.0
}
user_result = engine.predict_all(**user_farm)
print(f"User-Defined Optimal Crop: {user_result['stage1_recommendation']['recommended_crop'].upper()} (Confidence: {user_result['stage1_recommendation']['confidence_percent']}%)")
print(f"User-Defined Yield: {user_result['stage2_yield_forecast']['yield_tonnes_per_ha']} t/ha | Total Harvest: {user_result['stage2_yield_forecast']['total_production_tonnes']} Tonnes")

# Batch user-defined prediction DataFrame
user_batch_df = pd.DataFrame([
    {'N': 85, 'P': 48, 'K': 40, 'temperature': 24.0, 'humidity': 84.0, 'ph': 6.4, 'rainfall': 240.0, 'Area_ha': 3.0},
    {'N': 20, 'P': 130, 'K': 200, 'temperature': 22.0, 'humidity': 92.0, 'ph': 6.0, 'rainfall': 110.0, 'Area_ha': 2.5},
    {'N': 40, 'P': 65, 'K': 80, 'temperature': 19.0, 'humidity': 16.0, 'ph': 7.2, 'rainfall': 80.0, 'Area_ha': 5.0}
])
batch_results = engine.predict_batch(user_batch_df)
batch_results
